# 객체 검출 / 탐지 (Object Detection)

- 분류 (Classification) : 이미지 한 장 -> 라벨 하나
- 객체 탐지 : 이미지 내 특정 [위치]에 객체를 찾고, 해당 객체를 [분류] / 이미지 1장 -> 박스 N개 + 분류 + 신뢰도
    - 출력의 개수가 정해져 있지 않음 / 어떤 영상은 박스가 1개, 어떤 영상은 박스가 5개
        - 신경망의 출력 노드 수는 고정 -> 정답의 개수가 가변 (구조적으로 까다로움)
    - 위치를 맞추는 **회귀** 문제와 **분류**를 수행하는 문제가 동시에 존재
    - 배경이 압도적으로 많음 (박스가 전체 이미지에 5%미만) -> 극단적 불균형 현상
- 라벨링 비용
    - 분류 라벨 : 이미지 1장 /영상 1장 당 1개
    - 검출 라벨 : 병변마다 박스를 그려야 함 -> 수십 배 시간이 소요
    - 검출 라벨이 붙어있는 의료 데이터가 매우 비싼 자원 

## Bounding Box 표현법 

- 같은 박스라도 서로 다른 방식으로 표현 / 라이브러리 별로 표현법이 다름
    - 코너 좌표 (픽셀 값을 정수) : X축과 Y축의 최소 지점 / 최대 지점 (Pascal VOD / Torchvision / 시각화)
    - 중심 크기 좌표 (중심점과 박스 길이) : X축과 Y축의 중심지점 + 높이 + 너비
    - 정규화 중심 크기 좌표 (이미지 대비 중심점과 박스의 길이) : 이미대비 X축과 Y축의 중심지점 + 이미지 대비 박스의 높이 + 이미지 대비 박스의 너비 (YOLO)
    - 좌상단 크기 좌표 (픽셀의 원점을 기준으로 박스의 길이) : X축의 최소 Y축 최소 (픽셀 원점 기준) + 박스의 너비와 높이
- 이미지 사이즈 마다 픽셀 좌표들은 상대적 위치를 표현함에 있어 서로 다른 기준을 가져가기 때문에 학습이 잘못 수행될 수 있음
    - 정규화 : 이미지의 좌표를 0~ 1 변환
    - 이미지 크기에 따라 고유 위치를 쉽게 찾아 학습하기 위함 

## 평가 지표 

- IoU (Intersection Over Union) : 예측 박스와 정답 박스가 얼마나 겹치는가
    - IoU = 교집합의 넓이 (겹치는 부분의 넓이) / 합집합의 넓이 (두 박스의 전체넓이)
    - 1 가까울 수록 두 박스가 서로 완전히 겹침 / 0 가까울 수 록 박스가 겹치지 않음
    - IoU Threshold : 50% (AP50)  / IoU >= 0.5 맞게 찾으로 판단 -> TP (True Positive)
        - 엄격한 경우 75% (AP75)
    - 평가 지표로도 사용 / 객체 탐지 알고리즘의 손실함수로도 사용
    - 판정 규칙 용어
        - TP : 정답 박스와 IoU가 기준 이상, 클래스도 일치
        - FP : IoU 미달, 클래스도 불일치 /  이미 다른 예측과 짝지어진 정답에 또 매칭된 박스를 중복하여 예측
        - FN : 어떤 예측과도 짝지어지지 않는 정답 박스
        - 정답 박스 하나에는 예측 하나만 TP인정
        - 만약 같은 물체에 박스 3개 그리면 -> 1 TP / 2 FP

- NMS (Non-Maximum Supperssion) : 같은 물체에 겹쳐 나온 중복 박스를 정리하는 후처리 작업
    - 탐지 모델이 기본적으로 한 물체 주변에 많은 Box 생성
    - 그대로 두면 중복 박스가 모두 FP로 처리되어 성능이 저하
    - NMS 지표
        1. 신뢰도 (Confidence Socre) : 예측한 박스안에 해당 클래스(물체)가 있을 확률
        2. 임계값 (NMS Threshold) : 신뢰도에 대한 임계값을 정해, 특정 신뢰도 미만으로 값이 떨어지는 Box는 삭제
    - NMS 절차
        1. 신뢰도가 너무 낮은 박스를 먼저 제거 (0.25)
        2. 남은 Box들의 신뢰도를 내림차순으로 정렬 -> 가장 신뢰도가 높은 박스를 선택
        3. 확정된 박스와 IoU가 NMS 임계값 이상인 박스를 제거 (같은 물체로 간주)
        4. 남은 Box가 없어질 때 까지 2,3을 반복
        5. 클래스 별로 수행

- AP / mAP (Mean Average Precision)
    - AP : 모델이 정답을 빠짐없이 찾으면서, 확신이 높은 예측일수록 실제로 맞는가를 수치로 나타낸 지표
    - 전체 이미지를 모두 활용해 계산 (한장의 이미지만으로는 계산하지 않음)
    - 검출에서 성능측정에 문제가 발생하는 경우
        - 아주 확실한 박스 1개만 생성 : 틀린건 거의 없으나, 대부분 놓침
        - 박스를 수백개 생성하는 경우 : 정답은 거의 다 찾지만, 중복답안이 많은 경우
    - 위 2가지 문제를 모두 해결했는가를 평가하는 지표로써 사용
    - 계산 : 예측을 신뢰도 순으로 정렬 -> TP/FP 판정 -> Precision-Recall 곡선 -> 곡선의 아래 면적 = AP
    - 전체 데이터에서 각 클래스 별 AP -> 모든 클래스에 대해 평균 mAP 

## 검출 모형의 계열 

1. 2-Stage 계열 모형 (먼저 후보를 찾고, 그 다음에 분류를 수행)
    - 작동 단계
        1. Region Proposal 탐색 : 물체가 있을 법한 후보지를 탐색
        2. Classification : 각 후보지를 분류하고 박스를 세부적으로 조정
    - 정확도가 높음 (느림) / 정밀도가 중요한 의료 검출에서 사용
    - R-CNN (2014) -> Faster R CNN

2. 1-Stage 계열 모형 (후보 추출없이 한번에 박스를 그리는 모형)
    - YOLO (You Only Look Once) : 이미지를 격자형태로 나누어 박스와 클래스를 동시에 예측하는 CNN 기반 모형 (실시간 처리)
    - 매우 속도가 빠르고 간단하기 때문에 Streaming Data (Live Data) 강점
    - SSD , RetianNet
      
3. Transformer 계열 (후처리 파트까지 신경망 내부에 구성)
    - DETR : 박스를 집합(Set)으로 묶어 한번에 예측 -> NMS가 필요 하지 않음  
    - D-FINE, RT-DETR : 실시간 처리가 가능하도록 개선

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

In [3]:
from PIL import Image
import tensorflow as tf
from keras.preprocessing import image

In [4]:
import matplotlib.pyplot as plt

In [5]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [6]:
import glob # 파일 명을 검색해서 매칭되는 경로 문자열 리스트를 반환하는 기능을 가진 함수들의 집합체 

In [10]:
det_path = 'data/04_Foraminal_Stenosis_Data/'

case_list = sorted(os.listdir(det_path)) # 각 환자 폴더를 리스트로 구성 
xml_list  = sorted(glob.glob(det_path + '*/*.xml')) # 각 환자 내  xml 주석파일을 호출 -> 모든 환자 폴더를 한단계씩 들어가 파일을 리스트로 출력 
png_list  = sorted(glob.glob(det_path + '*/*.png')) # 각 환자 내 존재하는 png 파일을 모두 호출 

print('환자 폴더 수    :', len(case_list))
print('전체 이미지 수  :', len(png_list))
print('주석(XML) 수    :', len(xml_list))

환자 폴더 수    : 500
전체 이미지 수  : 6644
주석(XML) 수    : 1474


- Pascal VOC XML 구조
- <filename> : 이미지 명 
- <size> : 해당 이미지의 사이즈 / width 가로 , height 높이 , depth 채널 (intensity)
- <object> : 박스 하나에 대한 정보 (여러개가 있다면 반복)
    - <name> : 클래스 명 -> 첫글자 : 좌/우 / 마지막숫자 : 협착 등급
    - <bndbox> : 이미지 내 Box 위치를 픽셀 좌표로 표현
    - <level> : 척추 분절 

In [9]:
print(open(xml_list[0], encoding='utf-8').read())

<?xml version='1.0' encoding='utf-8'?>
<annotation>
  <folder>DatasetV0.28_ControlV2</folder>
  <filename>IM000002.png</filename>
  <path>Foramina_Detection\0001\IM000002.png</path>
  <source>
    <database>Unknown</database>
  </source>
  <size>
    <width>512</width>
    <height>512</height>
    <depth>1</depth>
  </size>
  <segmented>0</segmented>
  <object>
    <name>RFS0</name>
    <pose>Unspecified</pose>
    <truncated>0</truncated>
    <difficult>0</difficult>
    <bndbox>
      <xmin>234</xmin>
      <ymin>320</ymin>
      <xmax>274</xmax>
      <ymax>373</ymax>
    </bndbox>
    <level>L4-L5</level>
  </object>
</annotation>


In [11]:
# XML 주석 데이터를 정형데이터로 평탄화 
# 1개의 Box가 하나의 Row가 되도록 데이터 프레임을 구성 
import xml.etree.ElementTree as ET

In [12]:
def parse_voc_xml(xml_path):
    # 파일 전체를 트리형태 구조로 쉽게 읽어 들일 수 있도록 데이터를 변환  
    root = ET.parse(xml_path).getroot()

    # 환자의 고유 ID 를 폴더 경로로부터 추출 
    case_id  = os.path.basename(os.path.dirname(xml_path))  # xml 파일의 상위 폴더 명을 호출 = 환자ID 
    slice_id = os.path.basename(xml_path).replace('.xml','')  #  파일 명 = 하나의 이미지 슬라이스 / 확장자를 제거하고 파일이름만 추출 
    png_path = xml_path.replace('.xml','.png') # 이미지 파일 경로을 변수로 선언 

    # 이미지 한개 파일의 정보를 추출 
    size  = root.find('size') # 이미지 사이즈를 추출 
    img_w = int(size.find('width').text) # 해당 이미지 사이즈에서 가로길이 추출 
    img_h = int(size.find('height').text)# 해당 이미지 사이즈에서 세로길이 추출 

    # 데이터 오류로 size 값에 0값이 기록된 파일 존재 / 
    # 해당 xml 파일에 매칭되는 png 파일을 호출해서 파일 자체에서 가로길이와 세로길이 추출 
    if (img_w == 0 or img_h == 0) and os.path.exists(png_path):
        img_w, img_h = Image.open(png_path).size

    # 각 이미지에 존재하는 박스를 가져와 행을 생성 
    row_list = [] # 각각의 Box 정보가 하나씩 추가 
    # 박스에 정보가 담긴 <object>구조를 모두 리스트로 반환하여 obj 매개변수로 선언 
    for obj in root.findall('object'):
        name   = obj.find('name').text  # 좌우 구분과 협착 등급 정보 추출 
        bndbox = obj.find('bndbox')     # 박스의 사이즈 추출 
        level  = obj.find('level')      # 척추 분절 정보 추출 

        row_list.append({
            # 식별 정보 (환자 또는 이미지 구분0)
            'case_id'  : case_id,
            'slice_id' : slice_id,
            'png_path' : png_path,
            # 이미지의 크기 
            'img_w'    : img_w,
            'img_h'    : img_h,
            # 라벨 정보 
            'name'     : name, # 좌우구분 + 협착 등급 
            'side'     : name[0],   # 좌우구분 /L R                 
            'grade'    : int(name[-1]),  # 협착 등급           
            'level'    : level.text if level is not None else None,    # 척추 분절 
            # 박스의 좌표 (픽셀 좌표)
            'xmin'     : int(bndbox.find('xmin').text),
            'ymin'     : int(bndbox.find('ymin').text),
            'xmax'     : int(bndbox.find('xmax').text),
            'ymax'     : int(bndbox.find('ymax').text),
        })
    return row_list

In [13]:
# XML 파일을 가져와 Box의 정보를 모두 모아서 정형데이터로 변환 
box_list = [] # 모든 환자의 Box 정보를 담을 리스트를 구성 
for xml_path in xml_list:
    box_list = box_list + parse_voc_xml(xml_path)

df_box = pd.DataFrame(box_list)
print(df_box.shape)
df_box.head(5)

(2979, 13)


,case_id,slice_id,png_path,img_w,img_h,name,side,grade,level,xmin,ymin,xmax,ymax
0,0001,IM000002,data/04_Foraminal_Stenosis_Data\0001\IM000002.png,512,512,RFS0,R,0,L4-L5,234,320,274,373
1,0001,IM000003,data/04_Foraminal_Stenosis_Data\0001\IM000003.png,512,512,RFS0,R,0,L2-L3,247,203,279,259
2,0001,IM000003,data/04_Foraminal_Stenosis_Data\0001\IM000003.png,512,512,RFS2,R,2,L3-L4,242,263,274,318
3,0001,IM000003,data/04_Foraminal_Stenosis_Data\0001\IM000003.png,512,512,RFS3,R,3,L4-L5,236,322,274,373
4,0001,IM000003,data/04_Foraminal_Stenosis_Data\0001\IM000003.png,512,512,RFS0,R,0,L5-S1,239,377,285,423


In [14]:
df_box.to_csv('data/04_Foraminal_Stenosis_Data/04_xml_bndbox.csv')

In [15]:
print('주석이 있는 이미지 수 :', df_box.groupby(['case_id','slice_id']).ngroups)
print('주석이 있는 환자 수   :', df_box['case_id'].nunique())
print('전체 박스 수          :', len(df_box))
print(df_box['name'].value_counts())

주석이 있는 이미지 수 : 1474
주석이 있는 환자 수   : 469
전체 박스 수          : 2979
name
LFS0    1057
RFS0     952
LFS1     272
RFS1     236
LFS2     139
RFS2     115
LFS3     115
RFS3      93
Name: count, dtype: int64


In [16]:
print(pd.crosstab(df_box['side'], df_box['grade']))
print()
print(df_box['level'].value_counts())
# df_box 데이터에서 빈도수 -> Box 개수 -> 불균형 

grade     0    1    2    3
side                      
L      1057  272  139  115
R       952  236  115   93

level
L3-L4    795
L2-L3    686
L4-L5    649
L1-L2    571
L5-S1    278
Name: count, dtype: int64


In [17]:
# 이미지 한 장에 박스가 몇개 있는지 확인 
box_per_image = df_box.groupby(['case_id','slice_id']).size()
print(box_per_image.describe())

count    1474.000000
mean        2.021031
std         1.037108
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         5.000000
dtype: float64


## 이미지 검출에서 품질 점검 

1. 주석은 있는데 이미지 파일이 없는경우  -> 학습에서 제외해야 함
2. 이미지의 크기가 각각 달라지는가 확인
3. 박스가 지나치게 작을 때 (Small Object)

In [18]:
# 1. 주석은 있는데 이미지 파일이 없는경우
cond_missing = ~df_box['png_path'].apply(os.path.exists)
print('이미지가 없는 박스 수 :', cond_missing.sum())
print(df_box.loc[cond_missing, ['case_id','slice_id']].drop_duplicates())

이미지가 없는 박스 수 : 1
     case_id  slice_id
1939    0329  IM000001


In [19]:
# 2. 이미지의 크기가 각각 달라지는가 확인
print(df_box[['img_w','img_h']].drop_duplicates().sort_values('img_w').to_string(index=False))

 img_w  img_h
   256    256
   320    320
   336    336
   352    352
   384    384
   432    432
   448    448
   480    480
   512    512
   560    560
   576    576
   672    672
   704    704
   720    720
   800    800
  1008   1008


In [21]:
# 3. 박스가 지나치게 작을 때 (Small Object)
df_box['box_w'] = df_box['xmax'] - df_box['xmin']
df_box['box_h'] = df_box['ymax'] - df_box['ymin']
# 전체 이미지 크기 대비 박스의 크기 비율 
df_box['area_ratio'] = (df_box['box_w'] * df_box['box_h']) / (df_box['img_w'] * df_box['img_h']) * 100

print(df_box[['box_w','box_h','area_ratio']].describe())

             box_w        box_h   area_ratio
count  2979.000000  2979.000000  2979.000000
mean     34.727090    44.159114     0.784008
std       9.772344    12.513292     0.141378
min      15.000000    20.000000     0.388755
25%      26.000000    33.000000     0.684366
50%      34.000000    43.000000     0.770950
75%      42.000000    52.000000     0.870117
max      64.000000    84.000000     1.371094


- 박스의 면적이 전체 이미지 대비 1% 채 안되는 수준 -> 평균 0.78% (Small Object Detection)
- Small Object Detection이 학습이 어려운 이유 
    - 1. CNN 알고리즘 특성 상, 해상도가 줄면서 특징을 얻어내는 구조 -> 이미지가 소실
    - 2. 입력된 데이터가 어떤 클래스인지 알고리즘이 찾지 못하는 현상


- Small Object Detection
- 통상 32x32 픽셀 미만 / 박스의 면적이 전체 이미지 면적에 1%이하
- CNN 알고리즘 깊은 층일수록 이미지의 의미 (Semantic) 정보는 풍부해지지만, 소형 객체는 1,2픽셀 미만으로 줄어들어 정보가 소실
- IoU(예측 박스와 정답 박스가 얼마나 잘 겹치는가)가 극도로 민감해짐
    - 큰 객체 200x200 : 예측의 오차 -> 10px -> IOU 90% (TP)
    - 작은 객체 20x20 : 예측의 오차 -> 10px -> IOU 33% (FP)
- 해결 :
    1. [데이터 관점] : 고 해상도 입력 (연산량 증가)
    2. [데이터 입력 관점] : 타일링 / SAHI (Slicing Aided Hyper Inference) : 이미지를 겹쳐 잘라서 병합한 뒤 추론하는 방법
        - 대부분 구분가능한 객체들이 비슷
    3. [신경망 구조] : FPN (Feature Pyramind Network) : 고해상도의 얇은 층 + 의미를 전달하는 깊은 층
    4. [손실 함수 / 평가 지표] : IoU 손실함수를 계산 -> NWD, DIoU, CIoU
    5. [데이터 배정 관점] : ATSS (동적 양성 샘플 배정) - 소행 객체의 학습 신호 